# Phase A-1/A-2: 路線価の変動分析（記述統計）

REX地価コンテンツデータセットを用いた年次変化率の記述的分析。  
**このノートブックは合成データで動作確認済み。実データはパスを変えるだけで適用可能。**

In [ ]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import japanize_matplotlib  # 日本語フォント（なければ pip install japanize-matplotlib）

from rex_io import load_and_prepare
from compute_change import (
    by_chikukbn,
    by_district_type,
    by_prefecture,
    donut_effect_table,
    price_distribution,
)

plt.rcParams["figure.dpi"] = 120

# ── データパス設定（ここだけ変えれば実データに切り替え可能）────────
DATA_PATH = Path("/Users/KASU/REX_data_2020-2022/2022/nouhin_line_2022.shp")         
# DATA_PATH = Path("/実データパス/nouhin_line_2022.shp")  # 実データ時はこちら

## 1. データ読み込み

In [ ]:
gdf = load_and_prepare(DATA_PATH)

print(f"件数     : {len(gdf):,}")
print(f"CRS      : {gdf.crs}")
print(f"年度     : {gdf['nendo'].unique()}")
print(f"列       : {list(gdf.columns)}")

## 2. 路線価の分布

In [ ]:
price_distribution(gdf)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左: 路線価の分布（対数スケール）
axes[0].hist(np.log10(gdf["kakaku"].clip(lower=1)), bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("log10(路線価) [千円/㎡]")
axes[0].set_ylabel("件数")
axes[0].set_title("路線価の分布（対数スケール）")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"10^{x:.0f}"))

# 右: 地区区分の割合
counts = gdf["chikukbn_label"].value_counts()
axes[1].barh(counts.index, counts.values, color="coral")
axes[1].set_xlabel("件数")
axes[1].set_title("地区区分別の件数")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 3. 年次変化率の分布

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
change_cols = [
    ("change_y1", "当年 vs 前年（2022→2021）"),
    ("change_y2", "前年 vs 前前年（2021→2020）"),
    ("change_2y", "当年 vs 前前年（2022→2020）"),
]
for ax, (col, title) in zip(axes, change_cols):
    data = gdf[col].dropna()
    ax.hist(data, bins=60, color="steelblue", edgecolor="white")
    ax.axvline(0, color="red", linestyle="--", linewidth=1.2, label="変化なし")
    ax.axvline(data.median(), color="orange", linestyle="-", linewidth=1.2,
               label=f"中央値: {data.median():.1f}%")
    ax.set_title(title)
    ax.set_xlabel("変化率 (%)")
    ax.legend(fontsize=8)

axes[0].set_ylabel("件数")
plt.suptitle("路線価の年次変化率の分布", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 4. 地区区分別の変化率

In [ ]:
# 集計テーブル（当年 vs 前年）
tbl = by_chikukbn(gdf)
tbl["change_y1"]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

order = [
    "ビル街地区", "高度商業地区", "繁華街地区",
    "普通商業・併用住宅地区", "普通住宅地区",
    "中小工場地区", "大工場地区",
]
medians = (
    gdf.groupby("chikukbn_label")["change_y1"]
    .median()
    .reindex(order)
    .dropna()
)
colors = ["#d73027" if v < 0 else "#4575b4" for v in medians]
bars = ax.barh(medians.index[::-1], medians.values[::-1], color=colors[::-1])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("変化率の中央値 (%)")
ax.set_title("地区区分別 路線価変化率の中央値（当年 vs 前年）")
plt.tight_layout()
plt.show()

## 5. Donut Effect の確認（都市中心からの距離帯別変化率）

In [ ]:
donut = donut_effect_table(gdf, city="東京")
donut

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (col, label) in zip(axes, [
    ("change_y1_median", "当年 vs 前年"),
    ("change_2y_median", "当年 vs 前前年（2年間）"),
]):
    vals = donut[col].dropna()
    colors = ["#d73027" if v < 0 else "#4575b4" for v in vals]
    ax.bar(range(len(vals)), vals, color=colors, edgecolor="white")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(vals.index, rotation=30, ha="right")
    ax.set_ylabel("変化率中央値 (%)")
    ax.set_title(f"東京中心からの距離帯別変化率（{label}）")

plt.suptitle("Donut Effect の検証（合成データ）", y=1.02)
plt.tight_layout()
plt.show()

## 6. Moran's I（空間的自己相関）

※ libpysal / esda が必要。未インストールの場合は `pip install libpysal esda` を実行。

In [ ]:
from spatial_autocorr import HAS_PYSAL, global_moran, moran_sensitivity, build_weights

if HAS_PYSAL:
    # 平面直角座標系へ変換（距離計算のため）
    gdf_proj = gdf.to_crs(epsg=3857)

    # Global Moran's I（k=8）
    result = global_moran(gdf_proj, col="change_y1")
    print("=== Global Moran's I（change_y1）===")
    for k, v in result.items():
        print(f"  {k}: {v}")

    # 感度分析（kを変えてMoran's Iがどう変わるか）
    print("\n=== 感度分析（k近傍数）===")
    sens = moran_sensitivity(gdf_proj, col="change_y1")
    print(sens)
else:
    print("libpysal / esda が未インストールです。pip install libpysal esda を実行してください。")